In [22]:
import math
import torch
import tiktoken
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

In [15]:
@dataclass
class GPTConfig:
    block_size: int = 1024 # GPT-2 max context length
    vocab_size: int = 50257 # 50,000 BPE merges + 256 byte tokens + 1 <|endoftext|>
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768 # embed_dim

In [93]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        # causal mask
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))

    def forward(self, x, past_key_value=None):
        B, T, C = x.size()
        
        # Calculate query, key, values for all heads in batch
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if past_key_value is not None:
            past_k, past_v = past_key_value
            k = torch.cat([past_k, k],dim=2).contiguous()
            v = torch.cat([past_v, v], dim=2).contiguous()
            
        # Store the updated keys and values for the next generation step    
        present_key_value = (k, v)

        T_total = k.size(2)
        # Attention (can also use F.scaled_dot_product_attention for speed)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,T_total - T:T_total, :T_total] == 0, -1e9)
        att = F.softmax(att, dim=-1)
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        
        return self.c_proj(y), present_key_value

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu    = nn.GELU(approximate='tanh') # GPT-2 uses tanh approximation
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x, past_key_value=None):
        attn_out, present_key_value = self.attn(self.ln_1(x), past_key_value=past_key_value)
        # Notice pre-norm architecture: LayerNorm happens BEFORE the layer
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return x, present_key_value

In [94]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        
        # GPT-2 ties the weights of the token embedding and the final linear layer
        self.transformer.wte.weight = self.lm_head.weight 

    def forward(self, idx, past_key_values=None):
        B, T = idx.size()

        past_length = past_key_values[0][0].size(2) if past_key_values is not None else 0
        
        pos = torch.arange(past_length, past_length + T, dtype=torch.long, device=idx.device)
        
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = tok_emb + pos_emb

        # Route x and the specific cache to each transformer block
        present_key_values = []
        for i, block in enumerate(self.transformer.h):
            # Fetch the cache for this specific block (if it exists)
            past_kv = past_key_values[i] if past_key_values is not None else None
            
            x, present_kv = block(x, past_key_value=past_kv)
            present_key_values.append(present_kv)
                    
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        return logits, present_key_values

    @classmethod
    def from_pretrained(cls, model_type):
        """Loads pretrained GPT-2 model weights from huggingface"""
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        from transformers import GPT2LMHeadModel
        print(f"Loading weights from pretrained gpt: {model_type}")

        # 1. Initialize our custom model based on the type
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_type]
        
        config_args['vocab_size'] = 50257
        config_args['block_size'] = 1024
        config = GPTConfig(**config_args)
        model = cls(config)
        sd = model.state_dict()
        sd_keys = [k for k in sd.keys() if not k.endswith('.attn.bias')] # Ignore causal mask buffers

        # 2. Initialize HuggingFace model
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()
        sd_keys_hf = [k for k in sd_hf.keys() if not k.endswith('.attn.masked_bias')]
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')]

        # 3. Copy weights while mapping and transposing where necessary
        # The weights for these specific layers need to be transposed because HF uses Conv1D
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        
        assert len(sd_keys_hf) == len(sd_keys), f"Mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # Transpose the HF weight and copy it over
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # Direct copy for standard layers (like LayerNorms and Embeddings)
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model

In [95]:
# Instantiate the model and load the 124M parameter weights!
model = GPT.from_pretrained('gpt2')

print("Successfully loaded GPT-2!")

Loading weights from pretrained gpt: gpt2


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 148/148 [00:00<00:00, 7003.28it/s]


Successfully loaded GPT-2!


## Inference

In [96]:
if torch.cuda.is_available():
    device = "cuda" # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = "mps"  # Apple Silicon (M1/M2/M3/M4)
else:
    device = "cpu"  # Standard CPU fallback

In [97]:
model.to(device)

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [98]:
def generate_text(model, prompt, max_new_tokens=50, temperature=0.8, top_k=40, device='cpu'):
    enc = tiktoken.get_encoding("gpt2")
    input_ids = enc.encode(prompt)
    
    idx = torch.tensor([input_ids], dtype=torch.long, device=device)
    model.eval()
    
    # Initialize the cache to None for the first pass
    past_key_values = None
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            
            # CRITICAL: If we have a cache, only pass the VERY LAST token
            # Otherwise, pass the whole sequence to build the initial cache
            idx_input = idx[:, -1:] if past_key_values is not None else idx
            
            # Forward pass: model now expects past_key_values and returns a tuple
            logits, past_key_values = model(idx_input, past_key_values=past_key_values)
            
            # We only care about the logits for the last time step
            logits = logits[:, -1, :] / temperature
            
            if top_k is not None:
                v_val, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                # Use -1e9 instead of -float('Inf') to prevent MPS NaN crashes
                logits[logits < v_val[:, [-1]]] = -1e9
                
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            
            # Append the new token to our running sequence for final decoding
            idx = torch.cat((idx, idx_next), dim=1)
            
    return enc.decode(idx[0].tolist())

In [111]:
generated = generate_text(model, "The wheel on the bus go round", device=device)

In [112]:
print(generated)

The wheel on the bus go round the corner, the bus has been pulled to the station. The driver goes round the corner, the driver goes round the corner.

The driver goes round the corner, the bus has been pulled to the station. The driver goes round the corner
